# 🥉 BRONZE Notebook 1 of 3
### Rio Mart: 10-minute grocery delivery

**Rule of Bronze: change nothing.**
No cleaning, no filtering, no renaming. Dump the files exactly as they arrived
and add two audit columns, which makes two things clear ie *when* a row landed and *which file* it came from.


```
  /Volumes (raw CSV)  →  🥉 BRONZE  →  🥈 SILVER  →  🥇 GOLD
       we are here
```

## Step 0 — One-time setup to create volume

It creates a schema called `rio_mart_quick_commerce` and a **Volume** called `raw`.

Then go to **Catalog → workspace → rio_mart_quick_commerce → Volumes → raw → Upload to this volume**
and upload the 5 CSV files.

In [0]:
CATALOG = "workspace"
SCHEMA  = "rio_mart_quick_commerce"
VOLUME  = "raw"

RAW_PATH = f"/Volumes/{CATALOG}/{SCHEMA}/{VOLUME}"

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{SCHEMA}")
spark.sql(f"CREATE VOLUME IF NOT EXISTS {CATALOG}.{SCHEMA}.{VOLUME}")
spark.sql(f"USE {CATALOG}.{SCHEMA}")

print("Upload CSV files here →", RAW_PATH)

Upload CSV files here → /Volumes/workspace/rio_mart_quick_commerce/raw


In [0]:
# Confirm the files actually landed in the Volume
import os

for file_name in sorted(os.listdir(RAW_PATH)):
    print(file_name)

dark_stores.csv
order_items.csv
orders.csv
products.csv
riders.csv


## Step 1 — Read one file

In [0]:
orders_raw = (
    spark.read
        .option("header", "true")
        .option("inferSchema", "true")
        .csv(f"{RAW_PATH}/orders.csv")
)

display(orders_raw.limit(10))

order_id,store_id,rider_id,customer_id,order_placed_ts,promised_minutes,delivered_ts,order_status,payment_mode,item_count,order_amount,discount_amount
ORD001047,ST010,R0120,CUST01963,2026-07-28T10:43:08.000Z,10,2026-07-28T10:56:14.000Z,delivered,WALLET,2,217.0,0.0
ORD005204,ST002,R0082,CUST03170,2026-08-19T08:49:38.000Z,10,2026-08-19T08:56:50.000Z,delivered,COD,3,1075.0,0.0
ORD001393,ST008,R0028,CUST00331,2026-07-30T11:05:29.000Z,10,2026-07-30T11:08:59.000Z,Delivered,WALLET,2,148.0,0.0
ORD001828,ST001,R0071,CUST03012,2026-08-01T21:48:19.000Z,10,2026-08-01T21:54:19.000Z,DELIVERED,upi,4,352.3,52.84
ORD003710,ST010,R0030,CUST01221,2026-08-11T10:08:58.000Z,10,2026-08-11T10:20:46.000Z,Delivered,WALLET,3,553.5,0.0
ORD005220,ST001,R0091,CUST00818,2026-08-19T18:13:06.000Z,10,2026-08-19T18:23:06.000Z,Delivered,card,3,260.55,0.0
ORD005175,ST004,R0024,CUST02889,2026-08-19T09:24:41.000Z,10,2026-08-19T09:32:17.000Z,DELIVERED,Wallet,4,347.1,0.0
ORD004003,ST008,R0038,CUST01409,2026-08-13T13:58:29.000Z,10,2026-08-13T14:10:29.000Z,Delivered,UPI,3,455.0,0.0
ORD003702,ST006,R0046,CUST02102,2026-08-11T22:14:27.000Z,10,2026-08-11T22:25:39.000Z,delivered,Card,3,875.0,0.0
ORD001864,ST004,R0004,CUST02639,2026-08-01T14:56:18.000Z,10,2026-08-01T15:03:54.000Z,Delivered,cod,2,297.0,44.55


## Step 2 — Land all 5 files in Bronze

Same three lines, five times. One small function keeps it readable.

In [0]:
from pyspark.sql import functions as F

def load_to_bronze(file_name: str, table_name: str):
    """Read one CSV from the Volume and land it in a Bronze Delta table."""
    df = (
        spark.read
            .option("header", "true")
            .option("inferSchema", "true")
            .csv(f"{RAW_PATH}/{file_name}")
            .withColumn("_source_file", F.lit(file_name))
            .withColumn("_ingested_at", F.current_timestamp())
    )
    df.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(table_name)
    print(f"{table_name:<22} {df.count():>7,} rows")

load_to_bronze("orders.csv",       "bronze_orders")
load_to_bronze("order_items.csv",  "bronze_order_items")
load_to_bronze("products.csv",     "bronze_products")
load_to_bronze("riders.csv",       "bronze_riders")
load_to_bronze("dark_stores.csv",  "bronze_dark_stores")

bronze_orders            5,762 rows
bronze_order_items      19,779 rows
bronze_products             60 rows
bronze_riders              120 rows
bronze_dark_stores          10 rows


In [0]:
%sql
SHOW TABLES

database,tableName,isTemporary
rio_mart_quick_commerce,bronze_dark_stores,false
rio_mart_quick_commerce,bronze_order_items,false
rio_mart_quick_commerce,bronze_orders,false
rio_mart_quick_commerce,bronze_products,false
rio_mart_quick_commerce,bronze_riders,false
rio_mart_quick_commerce,gold_category_sales,false
rio_mart_quick_commerce,gold_city_performance,false
rio_mart_quick_commerce,gold_rider_scorecard,false
rio_mart_quick_commerce,silver_order_items,false
rio_mart_quick_commerce,silver_orders,false


## Step 3 — Reality check: how dirty is this data?

Nothing is clean in Bronze. This is the moment the Silver layer comes into picture

In [0]:
%sql
-- Problem 1: the same status, spelled seven different ways
SELECT order_status, count(*) AS orders
FROM bronze_orders
GROUP BY order_status
ORDER BY orders DESC

order_status,orders
Delivered,1389
delivered,1377
DELIVERED,1345
Delivered,1318
cancelled,113
CANCELLED,113
Cancelled,107


In [0]:
%sql
-- Problem 2: duplicate orders (retries and replays from the source system)
SELECT
  count(*)           AS total_rows,
  count(DISTINCT order_id)    AS unique_orders,
  count(*) - count(DISTINCT order_id) AS duplicate_rows
FROM bronze_orders

total_rows,unique_orders,duplicate_rows
5762,5632,130


In [0]:
%sql
-- Problem 3: junk test orders, and orders with no rider attached
SELECT
  sum(CASE WHEN order_amount <= 0 THEN 1 ELSE 0 END) AS junk_orders,
  sum(CASE WHEN rider_id IS NULL  THEN 1 ELSE 0 END) AS orders_without_rider
FROM bronze_orders

junk_orders,orders_without_rider
27,45


In [0]:
%sql
-- Problem 4: one city, written five different ways
SELECT city, count(*) AS stores
FROM bronze_dark_stores
GROUP BY city
ORDER BY city

city,stores
BENGALURU,1
Bengaluru,1
DELHI,1
Delhi,1
Hyderabad,1
Mumbai,2
bengaluru,1
hyderabad,1
mumbai,1


## ✅ Bronze is done

| Table | What it holds |
|---|---|
| `bronze_orders` | every order, duplicates and junk included |
| `bronze_order_items` | the lines inside each order |
| `bronze_products` | product catalogue |
| `bronze_riders` | delivery riders |
| `bronze_dark_stores` | the mini-warehouses orders leave from |

Got 4 data quality problems and **fixed none of them**. Will fix them in silver

➡️ **Next: open `02_silver_layer`.**